<a href="https://colab.research.google.com/github/Garcchi-Prog/Lab1EDDII/blob/main/Lab1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
## Imports ##

from typing import Optional, Tuple
from graphviz import Digraph
from IPython.display import SVG
import csv


import pandas as pd

df = pd.read_csv('dataset_courses_with_reviews.csv', dtype=str)
df = df.fillna('0')  # Reemplaza vacíos con 0
indice_csv = {row['id']: row.tolist() for _, row in df.iterrows()}

In [ ]:
## Clase Nodo ##

class Node:

    def __init__(self, data: list[str]) -> None:
        self.data = data
        self.satis = self.satisfaccion(self.data[3], self.data[11], self.data[12], self.data[13], self.data[4])
        self.izquierda: Optional["Node"] = None
        self.derecha: Optional["Node"] = None
        self.altura=1

    def satisfaccion(self, rating, positive_reviews, negative_reviews, neutral_reviews, num_reviews):
        try:
            rating = float(rating)
            positive_reviews = float(positive_reviews)
            negative_reviews = float(negative_reviews)
            neutral_reviews = float(neutral_reviews)
            num_reviews = float(num_reviews)

            if num_reviews != 0:
                val = rating * 0.7 + ((5 * positive_reviews + negative_reviews + 3 * neutral_reviews) / num_reviews) * 0.3
                return round(val, 5)
            else:
                return 0.0
        except ValueError:
            return 0.0

In [ ]:
## Clase Arbol ##

class Tree:
    def __init__(self, raiz: Optional["Node"] = None) -> None:
        self.raiz = raiz
    
    

        

    def add_node(self, node_id: str) -> str:
        fila = indice_csv.get(node_id)

        if fila is None:
            return "La id del curso no existe en el dataset"

        newNode = Node(fila)

        if self.raiz is None:
            self.raiz = newNode
            return "Nodo raíz añadido exitosamente!"

        current = self.raiz
        parent = None

        while current is not None:
            parent = current
            if newNode.satis > current.satis:
                current = current.derecha
            elif newNode.satis < current.satis:
                current = current.izquierda
            else:
                return "Error! No pueden existir 2 nodos con el mismo valor de satisfacción!"

        if newNode.satis > parent.satis:
            parent.derecha = newNode
        else:
            parent.izquierda = newNode

        return "Nodo añadido exitosamente!"

    def search_node(self, dataSearch: str, option: str) -> Optional["Node"]:
        # option: "id" para buscar por ID, "satis" para buscar por satisfacción

        if option == "id":
            return self._search_by_id(dataSearch, self.raiz)

        elif option == "satis":
            current = self.raiz
            satis_buscada = float(dataSearch)
            while current is not None:
                if satis_buscada == current.satis:
                    return current
                elif satis_buscada > current.satis:
                    current = current.derecha
                else:
                    current = current.izquierda
            return None

        return None

    def _search_by_id(self, node_id: str, nodo: Optional["Node"]) -> Optional["Node"]:
        # Búsqueda recursiva por ID
        if nodo is None:
            return None
        if nodo.data[0] == node_id:
            return nodo
        izq = self._search_by_id(node_id, nodo.izquierda)
        if izq:
            return izq
        return self._search_by_id(node_id, nodo.derecha)

    def delete_node(self, dataSearch: str, option: str) -> str:
        searchedNode = self.search_node(dataSearch, option)

        if searchedNode is None:
            return "El nodo no fue encontrado, revise la información proporcionada."

        self.raiz = self._delete_recursivo(self.raiz, searchedNode.satis)
        return "Nodo eliminado exitosamente!"

    def _delete_recursivo(self, nodo: Optional["Node"], satis: float) -> Optional["Node"]:
        if nodo is None:
            return None

        if satis < nodo.satis:
            nodo.izquierda = self._delete_recursivo(nodo.izquierda, satis)
        elif satis > nodo.satis:
            nodo.derecha = self._delete_recursivo(nodo.derecha, satis)
        else:
            # Nodo encontrado — 3 casos
            if nodo.izquierda is None:
                return nodo.derecha
            elif nodo.derecha is None:
                return nodo.izquierda
            else:
                # Tiene 2 hijos: reemplazar con el sucesor (mínimo del subárbol derecho)
                sucesor = self._minimo(nodo.derecha)
                nodo.data  = sucesor.data
                nodo.satis = sucesor.satis
                nodo.derecha = self._delete_recursivo(nodo.derecha, sucesor.satis)

        return equilibrar(nodo)

    def _minimo(self, nodo: "Node") -> "Node":
        while nodo.izquierda is not None:
            nodo = nodo.izquierda
        return nodo
    
## Codigo AVL ## 

def rotacion_simple_derecha(nodo):
    #La raiz de la rotacion es el hijo izquierdo del nodo desbalanceado
    n_raiz = nodo.izquierda

    #La informacion de la derecha de la nueva raiz pasa a ser la izquierda del nodo
    nodo.izquierda = n_raiz.derecha

    #El nodo desbalanceado pasa a ser la derecha de la nueva raiz
    n_raiz.derecha = nodo

    return n_raiz

def rotacion_simple_izquierda(nodo):
    #La raiz de la rotacion es el hijo derecho del nodo desbalanceado
    n_raiz = nodo.derecha

    #La informacion de la izquierda de la nueva raiz pasa a ser la derecha del nodo
    nodo.derecha = n_raiz.izquierda

    #El nodo desbalanceado pasa a ser la izquierda de la nueva raiz
    n_raiz.izquierda = nodo

    return n_raiz

def rotacion_doble_izquierda_derecha(nodo):
    #Se hace una rotacion simple a la izquierda del hijo izquierdo del nodo desbalanceado
    nodo.izquierda = rotacion_simple_izquierda(nodo.izquierda)

    #Luego se hace una rotacion simple a la derecha del nodo desbalanceado
    n_raiz= rotacion_simple_derecha(nodo)

    return n_raiz

def rotacion_doble_derecha_izquierda(nodo):
    #Se hace una rotacion simple a la derecha del hijo derecho del nodo desbalanceado
    nodo.derecha = rotacion_simple_derecha(nodo.derecha)

    #Luego se hace una rotacion simple a la izquierda del nodo desbalanceado
    n_raiz= rotacion_simple_izquierda(nodo)

    return n_raiz

def equilibrar(nodo):
    if nodo is None:
        return nodo

    #Se hace el calculo del equilibrio del nodo 
    equilibrio = obtener_equilibrio(nodo)

    #Caso de desbalanceo (2)
    if equilibrio > 1:
        #Si el hijo izquierdo tiene equilibrio 0 o 1 solamente se hace simple
        if obtener_equilibrio(nodo.izquierda) >= 0:
            return rotacion_simple_derecha(nodo)

        #Si el hijo izquierdo tiene equilibrio -1 se aplica rotacion doble
        else:
            return rotacion_doble_izquierda_derecha(nodo)

    #Caso de desbalanceo (-2)
    if equilibrio < -1:
        #Si el hijo derecho tiene equilibrio 0 o -1 solamente se hace simple
        if obtener_equilibrio(nodo.derecha) <= 0:
            return rotacion_simple_izquierda(nodo)

        #Si el hijo derecho tiene equilibrio 1 se aplica rotacion doble
        else:
            return rotacion_doble_derecha_izquierda(nodo)

    return nodo

def obtener_equilibrio(nodo):
    if nodo is None:
        return 0

    #Se hace un llamado para calcular la altura de los hijos del nodo e inmediatamente se calcula el equilibrio 
    return altura(nodo.derecha) - altura(nodo.izquierda)

def factor_equilibrio(nodo):
    return obtener_equilibrio(nodo)

def altura(nodo):
    if nodo is None:
        return 0

    #Se aplica recursion para calcular la altura de los hijos del nodo
    return 1 + max(altura(nodo.izquierda), altura(nodo.derecha))

#CODIGO DE LAS OPERACIONES PARA BUSCAR PADRE, ABUELO Y TIO (RECURSIVO)

def BuscarPadre( raiz, nodo):
    if raiz is None:
        return None
    
    # Verificamos si  la raiz es el padre del nodo buscado tanto en la izquierda como en la derecha
    if (raiz.izquierda is  nodo) or (raiz.derecha is nodo):
        return raiz
    
    #Se aplica recursion  para buscar el padre en el subarbol izquierdo 
    padre_izquierda = BuscarPadre(raiz.izquierda, nodo)
    if padre_izquierda is not None:
        return padre_izquierda
    
    #Se aplica recursion para buscar el padre en el subarbol derecho
    padre_derecha = BuscarPadre(raiz.derecha, nodo)
    if padre_derecha is not None:
        return padre_derecha

    #Si no se encuentra el nodo en ninguno de los subarboles, se devuelve none
    return None

def BuscarAbuelo ( raiz, nodo):
    if raiz is None:
        return None
    
    #Se busca el padre del nodo dado
    padre = BuscarPadre(raiz, nodo)
    if padre is None:
        return None
    
    #Se busca el abuelo utilizando la funcion BuscarPadre, solamente que el nodo a buscar sera el del padre 
    abuelo=BuscarPadre(raiz, padre)
    return abuelo

def BuscarTio ( raiz, nodo):
    if raiz is None:
        return None

    #Se busca al padre para tener una referencia para buscar el abuelo
    padre = BuscarPadre(raiz, nodo)
    #Si el padre no existe, no se puede encontrar al tio
    if padre is None:
        return None
    
    #Se busca el abuelo para tener una referencia para localizar al tio
    abuelo = BuscarPadre(raiz, padre)
    #Si el abuelo no existe, no se puede encontrar al tio
    if abuelo is None:
        return None
    
    #Caso 1: El hijo izquierdo del abuelo es el padre, por lo tanto el hijo derecho sera el tio
    if abuelo.izquierda is padre:
        return abuelo.derecha
    #Caso 2: El hijo derecho del abuelo es el padre, por lo tanto el hijo izquierdo sera el tio
    elif abuelo.derecha is padre:
        return abuelo.izquierda
    #Caso 3: El padre no es hijo del abuelo, por lo cual se retorna None
    else:
        return None
    
#Codigo para obtener el nivel de un nodo

def obtener_nivel(raiz, nodo):
    if raiz is None:
        return -1

    if raiz is nodo:
        return 0

    nivel_izquierda = obtener_nivel(raiz.izquierda, nodo)
    if nivel_izquierda != -1:
        return nivel_izquierda + 1
        
    nivel_derecha = obtener_nivel(raiz.derecha, nodo)
    if nivel_derecha != -1:
        return nivel_derecha + 1